# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
!git clone https://github.com/nimra231/flyrank-ml-internship.git
%cd flyrank-ml-internship



Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 83 (delta 29), reused 31 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 1.64 MiB | 15.84 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/flyrank-ml-internship


#1:MY RULE:
Flag pages that are stale (not updated in 180+ days) AND still receiving meaningful visibility (impressions_90d ≥ 500), and separately check if they also suffer from low click-through rate. These pages have real audience exposure but outdated or underperforming content — the highest-value candidates for review.
Reason codes: stale_visible_low_ctr, stale_visible_page, low_ctr_visible_page, monitor_only
Signal 1 (flag-linked — behind FlyRank's refresh flags): staleness vs. decline rate — checking if staler pages are actually more likely to be declining.
Signal 2 (flag-linked — behind CTR-fix logic): CTR vs. position tier — checking if CTR actually drops as position worsens, as expected.

In [7]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates(subset="content_id")

# Signal 1: staleness vs decline
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=[0,90,180,365,10000],
                                  labels=["0-90d","90-180d","180-365d","365d+"])
signal1 = df.groupby("staleness_bucket").agg(
    n=("content_id","count"),
    pct_declining=("trend_direction", lambda x: (x=="down").mean())
)
print("SIGNAL 1: Staleness vs Decline Rate")
print(signal1)

# Signal 2: CTR vs position tier
signal2 = df.groupby("position_tier").agg(
    n=("content_id","count"),
    mean_ctr=("ctr","mean")
)
print("\nSIGNAL 2: CTR by Position Tier")
print(signal2)

SIGNAL 1: Staleness vs Decline Rate
                      n  pct_declining
staleness_bucket                      
0-90d             20655       0.512031
90-180d            9171       0.611057
180-365d            169       0.467456
365d+                 5       0.600000

SIGNAL 2: CTR by Position Tier
                   n  mean_ctr
position_tier                 
deep            1319  0.150212
page_1         11814  0.652467
page_3_5        7242  0.222484
striking        7304  0.323239
top_3           2321  1.483611


/tmp/ipykernel_1227/4159278762.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal1 = df.groupby("staleness_bucket").agg(


Signal 1 verdict: MIXED. Decline rate rises from 51.2% (0-90d) to 61.1% (90-180d), supporting the staleness hypothesis initially. But it drops to 46.7% at 180-365d and the 365d+ bucket has only 5 rows — too small to trust. The pattern isn't clean/monotonic, so I'm treating staleness as a weak, not strong, signal in my rule.

Signal 2 verdict: MIXED. CTR does drop sharply for 'deep' positions (0.150) versus better tiers, supporting the general idea. But the tiers don't decline smoothly — page_1 CTR (0.652) is lower than top_3 (1.484), and top_3's value being above 1.0 suggests either a data anomaly or a different CTR calculation basis for that tier, which I can't fully explain here. I'm treating position as directionally useful but not precisely reliable


##Rule encoded below:
 score=count of (stale, visible, low CTR) conditions met. One reason code per page based on which conditions triggered. Action label derived from the reason code

In [8]:
df["stale"] = df["days_since_last_update"] >= 180
df["visible"] = df["impressions_90d"] >= 500
df["low_ctr"] = df["ctr"] < 0.02

def assign_reason(row):
    if row["stale"] and row["visible"] and row["low_ctr"]:
        return "stale_visible_low_ctr"
    elif row["stale"] and row["visible"]:
        return "stale_visible_page"
    elif row["visible"] and row["low_ctr"]:
        return "low_ctr_visible_page"
    else:
        return "monitor_only"

def assign_action(reason):
    if reason in ["stale_visible_low_ctr", "stale_visible_page"]:
        return "refresh_now"
    elif reason == "low_ctr_visible_page":
        return "review_soon"
    else:
        return "monitor"

df["baseline_action_score"] = df["stale"].astype(int) + df["visible"].astype(int) + df["low_ctr"].astype(int)
df["reason_code"] = df.apply(assign_reason, axis=1)
df["action"] = df["reason_code"].apply(assign_action)

ranked = df.sort_values(["baseline_action_score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)
output_cols = ["content_id","baseline_action_score","reason_code","action",
               "days_since_last_update","impressions_90d","ctr","avg_position","trend_direction"]
ranked[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(ranked)} rows to work/outputs/baseline_action_score.csv")
ranked[output_cols].head(20)

Wrote 30000 rows to work/outputs/baseline_action_score.csv


,content_id,baseline_action_score,reason_code,action,days_since_last_update,impressions_90d,ctr,avg_position,trend_direction
0,content_5feee3994adb,3,stale_visible_low_ctr,refresh_now,194,7812,0.01,39.0,down
1,content_b16bd7307b39,3,stale_visible_low_ctr,refresh_now,194,4590,0.00,31.0,down
2,content_074ba6ead17b,3,stale_visible_low_ctr,refresh_now,183,533,0.00,48.0,down
3,content_a023517539fe,2,low_ctr_visible_page,review_soon,20,214047,0.01,85.8,up
4,content_c8e9d6ab9013,2,low_ctr_visible_page,review_soon,104,208678,0.00,9.7,down
5,content_453722754fea,2,low_ctr_visible_page,review_soon,20,140079,0.01,7.6,down
6,content_e752a4e03dd3,2,low_ctr_visible_page,review_soon,104,130892,0.01,23.9,down
7,content_54baba704595,2,low_ctr_visible_page,review_soon,104,130617,0.01,47.0,down
8,content_124763d39ca5,2,low_ctr_visible_page,review_soon,104,129803,0.01,33.2,down
9,content_4a6607efcb46,2,low_ctr_visible_page,review_soon,104,128068,0.01,2.2,up


## 3. Top-20 review



In [9]:
for i, row in ranked[output_cols].head(20).iterrows():
    print(f"{i+1}. {row['content_id']} | action={row['action']} | reason={row['reason_code']} | "
          f"stale={row['days_since_last_update']}d | impressions={row['impressions_90d']} | ctr={row['ctr']:.3f}")

1. content_5feee3994adb | action=refresh_now | reason=stale_visible_low_ctr | stale=194d | impressions=7812 | ctr=0.010
2. content_b16bd7307b39 | action=refresh_now | reason=stale_visible_low_ctr | stale=194d | impressions=4590 | ctr=0.000
3. content_074ba6ead17b | action=refresh_now | reason=stale_visible_low_ctr | stale=183d | impressions=533 | ctr=0.000
4. content_a023517539fe | action=review_soon | reason=low_ctr_visible_page | stale=20d | impressions=214047 | ctr=0.010
5. content_c8e9d6ab9013 | action=review_soon | reason=low_ctr_visible_page | stale=104d | impressions=208678 | ctr=0.000
6. content_453722754fea | action=review_soon | reason=low_ctr_visible_page | stale=20d | impressions=140079 | ctr=0.010
7. content_e752a4e03dd3 | action=review_soon | reason=low_ctr_visible_page | stale=104d | impressions=130892 | ctr=0.010
8. content_54baba704595 | action=review_soon | reason=low_ctr_visible_page | stale=104d | impressions=130617 | ctr=0.010
9. content_124763d39ca5 | action=revie

Top-20 review:

1: content_5feee3994adb — refresh_now (stale_visible_low_ctr). 194d stale, 7,812 impressions, CTR 0.010. High confidence. Would be wrong if the update date field is stale/incorrect in our source data.

2: content_b16bd7307b39 — refresh_now (stale_visible_low_ctr). 194d stale, CTR 0.000 (zero clicks). High confidence — zero CTR at real impression volume is a strong signal. Would be wrong if this page is intentionally non-clickable (e.g. an image/PDF result).

3: content_074ba6ead17b — refresh_now (stale_visible_low_ctr). Only 533 impressions — barely above our 500 threshold. Low-medium confidence. Would be wrong if impressions dip slightly next period and it no longer qualifies as "visible" at all.

4: content_a023517539fe — review_soon (low_ctr_visible_page). Only 20d since update, 214,047 impressions, CTR 0.010. High visibility but very fresh — flagged purely on CTR. Would be wrong if this ranks for a broad/informational query where low CTR is normal (e.g. answered directly in a featured snippet).

5: content_c8e9d6ab9013 — review_soon (low_ctr_visible_page). 104d stale, 208,678 impressions, CTR 0.000. Medium-high confidence given the volume. Would be wrong if tracking/analytics broke for this page rather than genuine zero clicks.

6:content_453722754fea — review_soon (low_ctr_visible_page). 20d stale, 140,079 impressions, CTR 0.010. Same pattern as #4 — recently touched but still low CTR. Would be wrong if it's a snippet-cannibalized query.

7:  content_e752a4e03dd3 — review_soon (low_ctr_visible_page). 104d stale, 130,892 impressions, CTR 0.010. Medium confidence. Would be wrong if this CTR is typical for its content category.

8:  content_54baba704595 — review_soon (low_ctr_visible_page). Same pattern, 130,617 impressions, CTR 0.010. Would be wrong for the same category-baseline reason as above.

9: content_124763d39ca5 — review_soon (low_ctr_visible_page). 129,803 impressions, CTR 0.010. Would be wrong if position is already top_3 (where our data showed anomalously high CTR expectations).

10:  content_4a6607efcb46 — review_soon (low_ctr_visible_page). 128,068 impressions, CTR 0.010. Would be wrong if this is a duplicate/near-duplicate of another ranked page absorbing its clicks.

11: content_39881853ef0c — review_soon (low_ctr_visible_page). 20d stale, 112,434 impressions. Fresh content with low CTR — would be wrong if it simply hasn't accumulated enough click data yet post-update.

12: content_109f8f7c9d39 — review_soon (low_ctr_visible_page). 90,476 impressions, CTR 0.010. Would be wrong if seasonal demand explains the dip.

13: content_32cfb0b2fccf — review_soon (low_ctr_visible_page). 89,361 impressions, CTR 0.010. Same reasoning as above — would be wrong if category-typical.

14: content_fb4bf6555c79 — review_soon (low_ctr_visible_page). 104d stale, CTR 0.000 on 84,093 impressions. Higher confidence given zero clicks at volume. Would be wrong if tracking failed.

15: content_a7c2dfc8a6ec — review_soon (low_ctr_visible_page). 20d stale, 76,868 impressions. Would be wrong if recently updated and CTR simply hasn't caught up yet.

16:content_013a5d3dfc26 — review_soon (low_ctr_visible_page). 65,652 impressions, CTR 0.010. Would be wrong if this is a low-intent informational query by nature.

17:content_d274ac4158ef — review_soon (low_ctr_visible_page). Only 26d stale, 65,138 impressions. Would be wrong for the same "too fresh to judge" reason as other 20-26d rows.

18:content_cf56e2e2e282 — refresh_now (stale_visible_page). 194d stale, CTR 0.150 — actually decent CTR, flagged purely on staleness+visibility, not CTR. Would be wrong if the content is evergreen and doesn't actually need updating despite its age.

19:content_7368877ea310 — refresh_now (stale_visible_page). 194d stale, CTR 0.130. Same reasoning as #18 — would be wrong if evergreen.

20:content_e5f459e737b7 — review_soon (low_ctr_visible_page). 20d stale, 56,363 impressions, CTR 0.010. Would be wrong if too recently updated to have stabilized CTR yet.




##Weakest picks:
 content_074ba6ead17b — flagged only because it's barely over the 500-impression visibility threshold (533); a small measurement fluctuation would flip its classification entirely, so I'm not fully confident in this one. content_a023517539fe — only 20 days since its last update, so calling it a "problem" this early may be premature; low CTR right after a content change could simply mean rankings/clicks haven't stabilized yet, not that the page is genuinely broken.

In [12]:
print("Leakage check:")
print("Features used in scoring: days_since_last_update, impressions_90d, ctr")
print("trend_direction used only for Section 1 signal reporting, NOT as a scoring input.")
print("No product flags present:", not any(c in df.columns for c in ["health_score","priority_score","action_type"]))

Leakage check:
Features used in scoring: days_since_last_update, impressions_90d, ctr
trend_direction used only for Section 1 signal reporting, NOT as a scoring input.
No product flags present: True
